# DSPy extract extras

In [7]:
input_pages = "../input/schede/*/*"
output_dir = "../output/schede"

In [8]:
from glob import glob

paths = glob(input_pages)
paths.sort()
len(paths)

170

In [9]:
import dspy


class TextExtractor(dspy.Signature):
    """Extract text from an image using OCR.
    Normalize text by removing line-break hyphens, and accented letters instead of apostrophes when appropriate.
    """

    page: dspy.Image = dspy.InputField(desc="Image of the page to extract text from")
    title: str = dspy.OutputField(desc="Title of the document")
    text: str = dspy.OutputField(desc="Extracted text from the document")


class OCRModule(dspy.Module):

    def __init__(self):
        self.extractor = dspy.Predict(TextExtractor)

    def forward(self, img: dspy.Image) -> tuple[str, str]:
        result = self.extractor(page=img)
        return result.title, result.text

In [10]:
import os

from dotenv import load_dotenv
import dspy


load_dotenv()

lm = dspy.LM(
    model='vertex_ai/gemini-2.5-flash',
    vertex_credentials=os.getenv('VERTEX_AI_CREDS'),
    temperature=1.0,
    max_tokens=65535,
)
dspy.configure(lm=lm)
#dspy.settings.configure(track_usage=True)

extractor = OCRModule()

In [11]:
test_pages = paths[:2]

[
    extractor(dspy.Image.from_file(page))
    for page in test_pages
]

[('MAIL',
  'Cosa fare dei 3212 pezzi di Pkmail arrivati in redazione? Per il momento sono sotto chiave, in attesa di essere passati allo scanner e digitalizzati. Le risposte? Ehm, stiamo convincendo Uno a darci un bit d\'aiuto... Noi dobbiamo ancora riprenderci dalla lettura di tutti i vostri messaggi. Volete proprio sapere la verità? Pkers, siete davvero grandi! Alcuni di voi hanno di certo studiato da "trituratori", ma questo vi fa onore. Di seguito pubblichiamo un\'analisi sragionata sulla vostra Pkmail. Qua e là spunta il nome di qualche fortunatissimo Pker estratto, in rappresentanza dei compagni, dalla fatata e aliena mano di Lyla.\n\nANALISI PERCENTUALE POSTA IN ENTRATA\n43,5% fax\n35,7% snail mail\n0,77% messaggi anonimi\n20% virtualmente in rete\n0,03% messaggi in bottiglia\nDi Carlo Vinzio di Borgomanero\n\n(Novara). La sua opera è visibile nell\'olomonitor di Uno, in basso a destra.\n\nTRA GLI ARGOMENTI (DI OGNI TIPO!) AVETE PARLATO DI:\n\n33% Olovideografia\nJacopo Mistrel

## Extract all pages

In [12]:
import os
from pathlib import Path
import json


os.makedirs(output_dir, exist_ok=True)


def extract_page(in_path: str) -> None:
    pkna_id = Path(in_path).parent.stem
    input_base = Path(in_path).stem
    output_path = Path(output_dir) / f"{pkna_id}_{input_base}.json"
    if output_path.exists():
        return

    img = dspy.Image.from_file(in_path)
    title, text = extractor(img)
    data = {
        "input_page": in_path,
        "title": title,
        "text": text,
    }
    with open(output_path, "w") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


extract_page(in_path=paths[0])

In [13]:
from tqdm.notebook import tqdm


for path in tqdm(paths, desc="Processing pages"):
    extract_page(in_path=path)

Processing pages:   0%|          | 0/170 [00:00<?, ?it/s]